<a href="https://colab.research.google.com/github/mutallimovaydin/Build_and_Query_Feature_Store_for_ML_Training_and_Serving/blob/main/Build_and_Query_a_Feature_Store_for_ML_Training_and_Serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!kaggle datasets download ealaxi/paysim1

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
paysim1.zip: Skipping, found more recently modified local copy (use --force to force download)


In [31]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [32]:
!unzip paysim1.zip

Archive:  paysim1.zip
replace PS_20174392719_1491204439457_log.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: PS_20174392719_1491204439457_log.csv  



In [33]:
df = pd.read_csv('/content/PS_20174392719_1491204439457_log.csv')

In [34]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [35]:
df.shape

(6362620, 11)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [37]:
df.isna().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [38]:
df.duplicated().sum()

np.int64(0)

In [39]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [40]:
print(df['nameOrig'].nunique())
print(df['nameDest'].nunique())
print("Fraud rate:", df['isFraud'].mean())
print("Fraud percentage:", df['isFraud'].mean() * 100)

6353307
2722362
Fraud rate: 0.001290820448180152
Fraud percentage: 0.12908204481801522


In [41]:
df['isFraud'].value_counts()

,count
isFraud,
0,6354407
1,8213


In [42]:
orig_repeat = df.groupby('nameOrig').size()
print(orig_repeat.value_counts())
print("Max təkrar sayı:", orig_repeat.max())

1    6344009
2       9283
3         15
Name: count, dtype: int64
Max təkrar sayı: 3


In [43]:
df['balance_mismatch'] = ((df['oldbalanceOrg'] - df['amount']).round(2)!= df['newbalanceOrig'].round(2)).astype(int)
print(df['balance_mismatch'].value_counts())

balance_mismatch
1    5125552
0    1237068
Name: count, dtype: int64


In [44]:
fraud_customers = df.loc[df['isFraud'] == 1, 'nameOrig'].unique()
clean_customers = (df.loc[~df['nameOrig'].isin(fraud_customers), 'nameOrig'].drop_duplicates().sample(4000, random_state=42))
selected_customers = set(fraud_customers) | set(clean_customers)

subset = df[df['nameOrig'].isin(selected_customers)].copy()

In [45]:
print("Original shape:", df.shape)
print("Subset shape:", subset.shape)

print("Original fraud rate:", df['isFraud'].mean())
print("Subset fraud rate:", subset['isFraud'].mean())

Original shape: (6362620, 12)
Subset shape: (12247, 12)
Original fraud rate: 0.001290820448180152
Subset fraud rate: 0.6706132113987099


**Entity and event timestamp**

In [46]:
base_date = pd.Timestamp("2024-01-01")
subset['event_timestamp'] = (base_date + pd.to_timedelta(subset['step'], unit='h'))
subset = subset.sort_values(['nameOrig', 'event_timestamp']).reset_index(drop=True)

subset[['nameOrig', 'step', 'event_timestamp']].head()

,nameOrig,step,event_timestamp
0,C1000036340,655,2024-01-28 07:00:00
1,C1000086512,95,2024-01-04 23:00:00
2,C1000331499,262,2024-01-11 22:00:00
3,C1000437286,284,2024-01-12 20:00:00
4,C1000484178,504,2024-01-22 00:00:00


In [47]:
step_agg = (
    subset.groupby(['nameOrig', 'step'])
    .agg({'amount': ['size', 'sum', 'last']})
    .reset_index()
)

step_agg.columns = ['nameOrig', 'step', 'step_txn_count', 'step_amount_sum', 'step_last_amount']

step_agg.head()

,nameOrig,step,step_txn_count,step_amount_sum,step_last_amount
0,C1000036340,655,1,253648.68,253648.68
1,C1000086512,95,1,33676.59,33676.59
2,C1000331499,262,1,2016790.84,2016790.84
3,C1000437286,284,1,181182.79,181182.79
4,C1000484178,504,1,3018810.85,3018810.85


In [48]:
g_step = step_agg.groupby('nameOrig')

step_agg['txn_count_so_far'] = g_step['step_txn_count'].cumsum().groupby(step_agg['nameOrig']).shift(1).fillna(0)
step_agg['total_sent_so_far'] = g_step['step_amount_sum'].cumsum().groupby(step_agg['nameOrig']).shift(1).fillna(0)

step_agg['avg_amount_so_far'] = np.where(
    step_agg['txn_count_so_far'] > 0,
    step_agg['total_sent_so_far'] / step_agg['txn_count_so_far'],
    0.0
)

step_agg.head()

,nameOrig,step,step_txn_count,step_amount_sum,step_last_amount,txn_count_so_far,total_sent_so_far,avg_amount_so_far
0,C1000036340,655,1,253648.68,253648.68,0.0,0.0,0.0
1,C1000086512,95,1,33676.59,33676.59,0.0,0.0,0.0
2,C1000331499,262,1,2016790.84,2016790.84,0.0,0.0,0.0
3,C1000437286,284,1,181182.79,181182.79,0.0,0.0,0.0
4,C1000484178,504,1,3018810.85,3018810.85,0.0,0.0,0.0


In [49]:
step_agg['last_txn_amount'] = g_step['step_last_amount'].shift(1).fillna(0)
step_agg['prev_step'] = g_step['step'].shift(1)

step_agg['hours_since_last_txn'] = (step_agg['step'] - step_agg['prev_step']).fillna(0)

In [50]:
feature_col_step = ['nameOrig', 'step', 'txn_count_so_far', 'total_sent_so_far', 'avg_amount_so_far', 'last_txn_amount', 'hours_since_last_txn']
subset = subset.merge(step_agg[feature_col_step], on=['nameOrig', 'step'], how='left')
subset.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_mismatch,event_timestamp,txn_count_so_far,total_sent_so_far,avg_amount_so_far,last_txn_amount,hours_since_last_txn
0,655,TRANSFER,253648.68,C1000036340,253648.68,0.0,C1958275811,0.00,0.00,1,0,0,2024-01-28 07:00:00,0.0,0.0,0.0,0.0,0.0
1,95,CASH_OUT,33676.59,C1000086512,33676.59,0.0,C1759363094,0.00,33676.59,1,0,0,2024-01-04 23:00:00,0.0,0.0,0.0,0.0,0.0
2,262,TRANSFER,2016790.84,C1000331499,2016790.84,0.0,C1778895918,0.00,0.00,1,0,0,2024-01-11 22:00:00,0.0,0.0,0.0,0.0,0.0
3,284,CASH_OUT,181182.79,C1000437286,9601.00,0.0,C1827183939,150236.44,331419.24,0,0,1,2024-01-12 20:00:00,0.0,0.0,0.0,0.0,0.0
4,504,CASH_OUT,3018810.85,C1000484178,3018810.85,0.0,C895750711,65126.98,3083937.83,1,0,0,2024-01-22 00:00:00,0.0,0.0,0.0,0.0,0.0


**Leakage**

In [51]:
first_step = step_agg.groupby('nameOrig').head(1)

assert (
    (first_step['txn_count_so_far'] == 0) &
    (first_step['avg_amount_so_far'] == 0) &
    (first_step['total_sent_so_far'] == 0) &
    (first_step['last_txn_amount'] == 0) &
    (first_step['hours_since_last_txn'] == 0)
).all(), "Test ugursuz oldu: İlk emeliyatda sifir olmayan deyerler var!"

print('Test 1 passed')

Test 1 passed


In [52]:
same_step = subset.groupby(['nameOrig','step'])[['txn_count_so_far', 'total_sent_so_far', 'avg_amount_so_far', 'last_txn_amount', 'hours_since_last_txn']].nunique()
assert (same_step <=1).all().all()

print('Test 2 passed')

Test 2 passed


In [53]:
import os
DATA_DIR = '/content'

feature_cols = ['nameOrig','event_timestamp','txn_count_so_far', 'total_sent_so_far', 'avg_amount_so_far', 'last_txn_amount', 'hours_since_last_txn','balance_mismatch']

feature_df = subset[feature_cols].copy()
feature_df.to_parquet(os.path.join(DATA_DIR,'transaction_features.parquet'),index = False)
feature_df.head()

,nameOrig,event_timestamp,txn_count_so_far,total_sent_so_far,avg_amount_so_far,last_txn_amount,hours_since_last_txn,balance_mismatch
0,C1000036340,2024-01-28 07:00:00,0.0,0.0,0.0,0.0,0.0,0
1,C1000086512,2024-01-04 23:00:00,0.0,0.0,0.0,0.0,0.0,0
2,C1000331499,2024-01-11 22:00:00,0.0,0.0,0.0,0.0,0.0,0
3,C1000437286,2024-01-12 20:00:00,0.0,0.0,0.0,0.0,0.0,1
4,C1000484178,2024-01-22 00:00:00,0.0,0.0,0.0,0.0,0.0,0


In [54]:
entity_df = subset[['nameOrig', 'event_timestamp', 'isFraud']].copy()
entity_df.to_parquet(os.path.join(DATA_DIR,'entity_df.parquet'),index = False)
entity_df.head()

,nameOrig,event_timestamp,isFraud
0,C1000036340,2024-01-28 07:00:00,1
1,C1000086512,2024-01-04 23:00:00,1
2,C1000331499,2024-01-11 22:00:00,1
3,C1000437286,2024-01-12 20:00:00,0
4,C1000484178,2024-01-22 00:00:00,1


Xüsusiyyətlərin Sənədləşdirilməsi (`Feature Documentation`)
Bu bölmədə məlumat (`pipeline`) yaratdığı hər bir xüsusiyyətin (`feature`) mənası, hesablanma məntiqi və nə üçün lazım olduğu izah olunur.

1. Obyekt və Zaman (Entity & Timestamp)
nameOrig — Əsas obyekt açarı. Göndərən müştərinin ID nömrəsini təmsil edir. Aşağıdakı bütün tarixi göstəricilər məhz bu açar (`nameOrig`) üzrə hesablanır.

event_timestamp — Xüsusiyyətlərin alınması zamanı istinad nöqtəsi olan nöqtəvi zaman (`point-in-time`). PaySim-dəki step sahəsi real vaxtı deyil, simulyasiya edilmiş saatları göstərir. Feast platformasının tələblərinə uyğunlaşmaq üçün o, tam tarix-saat formatına çevrilir (`base_date + step saatları`). İstinad tarixi şərtidir və real əməliyyat tarixini əks etdirmir — burada yalnız zamanın ardıcıllığı (`chronological order`) önəmlidir.

2. Tarixi Göstəricilər (Point-in-Time Safe Features)
Aşağıdakı xüsusiyyətlər tamamilə yalnız əvvəlki addımların məlumatları əsasında hesablanır. Eyni göndərənə məxsus bir neçə əməliyyat eyni step (saat) daxilində baş verə biləcəyi üçün, məlumatlar əvvəlcə (`nameOrig, step`) səviyyəsində qruplaşdırılır və daha sonra fərdi əməliyyatlara birləşdirilir. Bu yanaşma eyni saatda baş verən əməliyyatların bir-birinin məlumatlarını "sızdırmasının" (data leakage) qarşısını alır — hər bir əməliyyat yalnız özündən əvvəl baş vermiş addımlardakı məlumatları "görə bilir".

txn_count_so_far — Göndərənin indiyə qədərki bütün əvvəlki addımlarda etdiyi ümumi əməliyyat sayı (cari addım nəzərə alınmır). Hesabın nə qədər köhnə və ya aktiv olduğunu göstərir; yeni və ya az aktiv hesablar daha yüksək fırıldaqçılıq riskinə malik ola bilər.

avg_amount_so_far — Göndərənin əvvəlki addımlardakı əməliyyat məbləğlərinin orta qiyməti. Cari əməliyyatın istifadəçinin adətkar olduğu tipik davranışdan kəskin şəkildə fərqlənib-fərqlənmədiyini aşkar etmək üçün istifadə olunur.

total_sent_so_far — Əməliyyat növündən asılı olmayaraq, göndərənin əvvəlki addımlardakı ümumi köçürmə məbləğlərinin kumulyativ cəmi. İstifadəçinin ümumi əvvəlki əməliyyat həcmini əks etdirir.

last_txn_amount — Göndərənin ən son əvvəlki addımındakı əməliyyat məbləği. Sonuncu məlum fəaliyyətə nisbətən qəfil sıçrayışları aşkar etməyə kömək edir. Qeyd: Əgər əvvəlki addımda bir neçə əməliyyat olubsa, daxili ardıcıllıq dəqiq məlum olmadığından təqribi olaraq həmin addımdakı sonuncu əməliyyat götürülür.

hours_since_last_txn — Göndərənin əvvəlki fərqli addımından etibarən keçən saatların (addımların) sayı. Fırıldaqçılıq əlaməti ola bilən qeyri-adi dərəcədə sıx tranzaksiya dalğalarını yaxalamaq üçün istifadə olunur.

3. Cari Hadisə Göstəricisi (Current-Event Feature)
balance_mismatch — Əməliyyatdan sonra göndərənin balansının (newbalanceOrig) gözlənilən balansla (oldbalanceOrg - amount) uyğun olub-olmadığını göstərən ikili (0/1) bayraq. Yuxarıdakı xüsusiyyətlərdən fərqli olaraq, bu, tarixi göstərici deyil — birbaşa cari əməliyyatın öz məlumatlarından əldə edilir. Gələcək məlumatlardan istifadə etmir, lakin "keçmiş" də deyil; bu, proqnoz zamanı hesablanan məlumat keyfiyyəti / cari hadisə siqnalıdır.

4. Hədəf Sütun (Label)
**kalın metin**isFraud — Proqnoz hədəfi. Yalnız entity_df.parquet faylına daxil edilib, xüsusiyyət faylında yer almır, çünki bu, xüsusiyyət deyil, məhz hədəf etikettidir.

Məlumat Sızmasının Qarşısının Alınması (Leakage Prevention Summary)
Tarixi xüsusiyyətlər addım səviyyəsində əməliyyat sayı və məbləğləri toplanaraq, cari addımın öz payı daxil olmaqla kumulyativ cəmi hesablandıqdan sonra həmin cari payın çıxılması yolu ilə əldə edilir. Nəticədə yalnız qəti şəkildə əvvəlki addımlar qalır. Bu yanaşma üç yoxlama ilə təsdiqlənmişdir:

Hər bir göndərənin ilk addımında bütün tarixi xüsusiyyətlər sıfıra bərabərdir.

Eyni (nameOrig, step) dəyərini paylaşan əməliyyatlar eyni tarixi xüsusiyyət qiymətlərinə malikdir (bu, eyni addım daxilində sızma olmadığını sübut edir).

Əl ilə hesablanmış xüsusiyyətlər dəsti (yalnız step < current_step şərtindən istifadə etməklə) nümunəvi müştəri üçün boru kəmərinin çıxışı ilə tam üst-üstə düşür.

**Seçmə Qeydi (Sampling Note)**
Dataset bütün dələduzluqla əlaqəli müştəriləri və təsadüfi olaraq seçilmiş 3,000 qeyri-dələduz müştərini əhatə edən səviyyəyə endirilmişdir. Hər bir seçilmiş müştərinin tam əməliyyat tarixçəsi saxlanılmışdır (tarixi xüsusiyyətlərin hesablanması üçün bu tələb olunur). Bu, etiket-şüurlu (label-aware) seçmə strategiyasıdır: buna görə də bu alt qrupdakı dələduzluq nisbəti orijinal populyasiyadakı təxminən ~0.129% dələduzluq səviyyəsini əks etdirmir. Alt qrup real dünya dələduzluq yayılmasını qiymətləndirmək üçün deyil, məhz xüsusiyyət boru kəmərinin hazırlanması və yoxlanılması üçün nəzərdə tutulub.